# 09 — Nulos y estado de gestión
En este dataset un nulo casi nunca es un dato perdido: es **estructural** (la columna no aplica
a esa materia, ese ámbito o ese tipo de fila). Este paso no limpia nada: clasifica cada columna
según por qué le faltan valores y cada proceso según cómo quedó al cierre de 2023.
Única columna nueva: `estado_gestion`.

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/nulos.ipynb
import matplotlib.pyplot as plt

OUT_EDA = CURATED / "eda"
OUT_FIG = REPORTS / "figures"
OUT_EDA.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED / "analitico" / "dataset_analitico_interno.parquet")
print(len(df), "filas,", len(df.columns), "columnas")
df["tipo_elemento_analitico"].value_counts()

## Porcentaje de nulos por columna y categoría de ausencia

In [ ]:
matriz = clasificar_matriz_nulos(df)
matriz.to_csv(OUT_EDA / "matriz_nulos_clasificada.csv", index=False, encoding="utf-8")
matriz[["columna", "n_nulos", "pct_nulos", "categoria_nulo"]].head(25)

In [ ]:
resumen_cat = matriz.groupby("categoria_nulo")["columna"].count().sort_values()
resumen_cat

In [ ]:
top = matriz.head(25).iloc[::-1]
colores = {"estructural_materia": "tab:blue", "estructural_geografico": "tab:orange", "estructural_recursos": "tab:green",
           "estructural_tipo_elemento": "tab:olive", "completa": "tab:gray", "otra_ausencia": "tab:gray"}
plt.figure(figsize=(10, 7))
plt.barh(top["columna"], top["pct_nulos"] * 100, color=[colores[c] for c in top["categoria_nulo"]])
plt.xlabel("% de filas nulas")
plt.title("Las 25 columnas con más nulos, coloreadas por causa de la ausencia")
plt.tight_layout()
plt.show()

## Estado de gestión de cada proceso
| estado | significa |
|---|---|
| `resolucion_total` | resolvió todo lo que atendió |
| `con_resolucion_parcial` | resolvió algo, menos de lo atendido |
| `en_tramite_exclusivo` | atendió causas y no resolvió ninguna en el año |
| `sin_movimiento` | cero causas atendidas |

In [ ]:
df["estado_gestion"] = clasificar_dinamica_procesal(df)
proc = df[df["tipo_elemento_analitico"] == "proceso"].copy()
conteo_estados = proc["estado_gestion"].value_counts(dropna=False)
pd.DataFrame({"procesos": conteo_estados, "pct": (conteo_estados / len(proc) * 100).round(1)})

In [ ]:
cruce = pd.crosstab(proc["materia_homologada"], proc["estado_gestion"], margins=True, margins_name="Total")
cruce.to_csv(OUT_EDA / "resumen_dinamica_procesal.csv", encoding="utf-8")
cruce

In [ ]:
pct = pd.crosstab(proc["materia_homologada"], proc["estado_gestion"], normalize="index") * 100
pct.plot(kind="barh", stacked=True, figsize=(11, 5), colormap="tab10")
plt.xlabel("% de los procesos de la materia")
plt.title("Estado de gestión dentro de cada materia")
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.35), ncol=4)
plt.tight_layout()
plt.show()

## Control contable: atendidas = resueltas + pendientes_fin
Una fila no cierra (5.2.2.1, p. 334, Oruro): es una discrepancia de la fuente y no se toca.

In [ ]:
con_datos = proc.dropna(subset=["resueltas", "pendientes_fin", "atendidas"])
diferencia = (con_datos["atendidas"] - (con_datos["resueltas"] + con_datos["pendientes_fin"])).abs()
print("filas verificadas:", len(con_datos), " descuadres:", int((diferencia > 0).sum()))
con_datos.loc[diferencia > 0, ["cuadro_origen", "pagina_pdf", "territorio", "materia_homologada", "tipo_proceso", "atendidas", "resueltas", "pendientes_fin"]]

## Figura del reporte

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
cat_counts = matriz["categoria_nulo"].value_counts()
ejes[0].barh(cat_counts.index, cat_counts.values, color="#2c3e50")
ejes[0].set_title("Columnas según naturaleza de la ausencia (" + str(len(matriz)) + " columnas)")
ejes[0].set_xlabel("columnas")
for i in range(len(cat_counts)):
    ejes[0].text(cat_counts.values[i] + 0.5, i, str(cat_counts.values[i]), va="center", fontweight="bold")
ejes[0].invert_yaxis()
ejes[1].pie(conteo_estados.values, labels=conteo_estados.index, autopct="%1.1f%%", startangle=140,
            colors=["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"], wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ejes[1].set_title("Estado de gestión de los procesos (N=" + str(len(proc)) + ")")
plt.tight_layout()
plt.savefig(OUT_FIG / "01_matriz_ausencias.png", dpi=200)
plt.show()